# Fake News Detection — End-to-End NLP Classification
**Industry:** Media & Publishing | **Type:** ML Classification (NLP) | **Target:** `label` (Real / Fake)

**Workflow:** Data → Statistics → EDA → Preprocessing → Feature Engineering → Modeling → Evaluation → Insights → Recommendations

> **Data setup:** download the Kaggle dataset (`emineyetm/fake-news-detection-datasets`) and place `Fake.csv` and `True.csv` in `../data/`.
> If they are missing, the notebook falls back to `../data/sample_news.csv` (a tiny **synthetic** file used only to smoke-test the code; its metrics are meaningless).

In [ ]:
import re, string, warnings, os, joblib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from collections import Counter

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay, roc_auc_score,
                             precision_recall_curve, classification_report)
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
DATA_DIR = "../data"
os.makedirs("../images", exist_ok=True); os.makedirs("../models", exist_ok=True)

# 1. Data Understanding

In [ ]:
def load_data(data_dir=DATA_DIR):
    fake_p, true_p = f"{data_dir}/Fake.csv", f"{data_dir}/True.csv"
    if os.path.exists(fake_p) and os.path.exists(true_p):
        fake = pd.read_csv(fake_p); fake["label"] = "Fake"
        true = pd.read_csv(true_p); true["label"] = "Real"
        print("Loaded full Kaggle dataset.")
        return pd.concat([fake, true], ignore_index=True)
    print("WARNING: Fake.csv / True.csv not found -> using SYNTHETIC sample (smoke test only).")
    return pd.read_csv(f"{data_dir}/sample_news.csv")

df = load_data()
df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)  # shuffle
df.head()

**Variable meaning**
| Column | Meaning | Type |
|---|---|---|
| `title` | Headline | Text |
| `text` | Article body | Text |
| `subject` | Section/category the article was filed under | Categorical |
| `date` | Publication date | Date (stored as string) |
| `label` | Target — `Real` / `Fake` | Categorical (binary) |

In [ ]:
print("Shape:", df.shape)
print("\nData types:\n", df.dtypes)
print("\nMissing values:\n", df.isna().sum())
print("\nEmpty/whitespace-only text:", (df["text"].fillna("").str.strip() == "").sum())
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate texts:", df["text"].duplicated().sum())
print("\nUnique values per column:\n", df.nunique())
print("\nNumerical columns:", df.select_dtypes("number").columns.tolist() or "none (raw data is all text; numerics are engineered later)")
print("Categorical columns:", df.select_dtypes("object").columns.tolist())

# 2. Statistical Analysis
The raw data is text, so we derive numeric descriptors (length, word count) to apply descriptive statistics.

In [ ]:
eda = df.copy()
eda["title"] = eda["title"].fillna(""); eda["text"] = eda["text"].fillna("")
eda["text_chars"] = eda["text"].str.len()
eda["text_words"] = eda["text"].str.split().str.len()
eda["title_words"] = eda["title"].str.split().str.len()
num_cols = ["text_chars", "text_words", "title_words"]

summary = eda[num_cols].describe().T
summary["median"] = eda[num_cols].median()
summary["mode"] = [eda[c].mode().iloc[0] for c in num_cols]
summary["variance"] = eda[num_cols].var()
summary["IQR"] = summary["75%"] - summary["25%"]
summary

In [ ]:
# Same statistics split by class -> the business-relevant view
by_class = eda.groupby("label")[num_cols].agg(["mean", "median", "std", "min", "max"])
by_class

In [ ]:
plt.figure(figsize=(5,4))
sns.heatmap(eda[num_cols].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation of numeric descriptors"); plt.tight_layout()
plt.savefig("../images/correlation_heatmap.png", dpi=120); plt.show()

**Interpretation (fill in after running on the full dataset):** compare mean vs median word count (skew), IQR (spread / typical article size) and class-wise differences. State whether length alone separates fake from real, and why high correlation between `text_chars` and `text_words` means we should not feed both to a linear model blindly.

# 3. Exploratory Data Analysis

## 3.1 Univariate — class balance, subject, length

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12,4))
sns.countplot(data=eda, x="label", ax=ax[0]); ax[0].set_title("Real vs Fake")
for p in ax[0].patches: ax[0].annotate(int(p.get_height()), (p.get_x()+p.get_width()/2, p.get_height()), ha="center", va="bottom")
sns.countplot(data=eda, y="subject", order=eda["subject"].value_counts().index, ax=ax[1]); ax[1].set_title("Articles per subject")
plt.tight_layout(); plt.savefig("../images/class_subject_distribution.png", dpi=120); plt.show()
print(eda["label"].value_counts(normalize=True).round(3))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12,4))
sns.histplot(eda["text_words"].clip(upper=eda["text_words"].quantile(0.99)), bins=50, kde=True, ax=ax[0]); ax[0].set_title("Word count distribution (99th pct clipped)")
sns.boxplot(data=eda, x="label", y="text_words", ax=ax[1]); ax[1].set_title("Word count by class")
plt.tight_layout(); plt.savefig("../images/length_distribution.png", dpi=120); plt.show()

## 3.2 Bivariate — subject vs label, length vs label

In [ ]:
ct = pd.crosstab(eda["subject"], eda["label"])
display(ct)
ct.plot(kind="bar", stacked=True, figsize=(9,4), title="Subject vs Label"); plt.tight_layout()
plt.savefig("../images/subject_vs_label.png", dpi=120); plt.show()
print("Subjects containing both classes:", int(((ct>0).sum(axis=1)==2).sum()), "of", len(ct))

> ⚠️ **Watch for leakage:** in the original Kaggle data the subject categories are largely disjoint between Real and Fake. If so, `subject` would let a model cheat; it is therefore **excluded from the model features** and only used for analysis.

In [ ]:
plt.figure(figsize=(6,5))
s = eda.sample(min(3000, len(eda)), random_state=RANDOM_STATE)
sns.scatterplot(data=s, x="title_words", y="text_words", hue="label", alpha=.5)
plt.title("Title length vs body length"); plt.tight_layout(); plt.savefig("../images/scatter_len.png", dpi=120); plt.show()

## 3.3 Word-frequency analysis (real vs fake)

In [ ]:
STOP = set(TfidfVectorizer(stop_words="english").get_stop_words())
def top_words(series, n=20):
    c = Counter()
    for t in series:
        c.update(w for w in re.findall(r"[a-z']+", t.lower()) if w not in STOP and len(w) > 2)
    return pd.DataFrame(c.most_common(n), columns=["word", "count"])

fig, ax = plt.subplots(1, 2, figsize=(13,6))
for a, lab in zip(ax, ["Real", "Fake"]):
    sns.barplot(data=top_words(eda.loc[eda.label==lab, "text"]), y="word", x="count", ax=a); a.set_title(f"Top words — {lab}")
plt.tight_layout(); plt.savefig("../images/top_words.png", dpi=120); plt.show()

## 3.4 Multivariate — class-wise profile of descriptors

In [ ]:
sns.pairplot(s[num_cols + ["label"]], hue="label", corner=True, plot_kws={"alpha":.4})
plt.savefig("../images/pairplot.png", dpi=100); plt.show()

**Document your EDA insights here** (class balance, subject overlap, length differences, distinctive vocabulary such as news-agency tags in real articles vs emotive/ALL-CAPS wording in fake ones).

# 4. Data Preprocessing

In [ ]:
data = df.copy()
n0 = len(data)

# Missing / inconsistent data
data["title"] = data["title"].fillna("")
data["text"] = data["text"].fillna("")
data = data[(data["text"].str.strip().str.len() > 0)]         # drop empty bodies
data = data.drop_duplicates(subset=["title", "text"])          # drop exact duplicates
data["label"] = data["label"].astype(str).str.strip().str.title()
data = data[data["label"].isin(["Real", "Fake"])]
print(f"Rows: {n0} -> {len(data)} after cleaning")

# Encode target (Fake = positive class = 1: the thing we want to catch)
data["y"] = (data["label"] == "Fake").astype(int)

# Outliers: extremely long articles are capped (not removed) to protect length-based features
cap = data["text"].str.split().str.len().quantile(0.99)
print("99th percentile word count used as cap:", cap)

In [ ]:
AGENCY_TAG = re.compile(r"^\s*[A-Z][A-Za-z .,/'-]{1,40}\(Reuters\)\s*-?\s*")   # e.g. 'WASHINGTON (Reuters) -'
def clean_text(t, strip_agency=True):
    if strip_agency: t = AGENCY_TAG.sub("", t)
    t = re.sub(r"http\S+|www\.\S+", " ", t)          # URLs
    t = re.sub(r"\S+@\S+", " ", t)                    # emails
    t = t.lower()
    t = re.sub(r"[^a-z\s]", " ", t)                    # keep letters only
    return re.sub(r"\s+", " ", t).strip()

data["content_raw"] = data["title"] + " . " + data["text"]
data["content"] = (data["title"] + " " + data["text"]).map(clean_text)
data[["content_raw", "content", "label"]].head(3)

**Why strip the agency tag?** Real articles in this dataset often start with a "CITY (Reuters) -" dateline. A model can reach near-perfect accuracy just by spotting it, which will not generalise to other publishers. We remove it and later **quantify the effect** (Section 9).

# 5. Feature Engineering

In [ ]:
def style_features(raw_title, raw_text):
    full = raw_title + " " + raw_text
    words = full.split()
    n_words = max(len(words), 1)
    letters = [c for c in full if c.isalpha()]
    return pd.Series({
        "n_words": min(len(raw_text.split()), cap),
        "title_len": len(raw_title.split()),
        "exclam_count": full.count("!"),
        "question_count": full.count("?"),
        "caps_ratio": sum(c.isupper() for c in letters) / max(len(letters), 1),
        "caps_word_ratio": sum(w.isupper() and len(w) > 1 for w in words) / n_words,
        "title_caps_ratio": sum(c.isupper() for c in raw_title if c.isalpha()) / max(sum(c.isalpha() for c in raw_title), 1),
        "avg_word_len": np.mean([len(w) for w in words]) if words else 0,
        "quote_count": full.count('"') + full.count("\u201c"),
        "digit_ratio": sum(c.isdigit() for c in full) / max(len(full), 1),
    })

style = data.apply(lambda r: style_features(r["title"], r["text"]), axis=1)
style["label"] = data["label"].values
style.groupby("label").mean().T.round(3)

**Why these help:** fake articles tend to use sensational punctuation (`!`, `?`), ALL-CAPS emphasis, and shorter/less formal text; real journalism uses more quotations, figures and neutral headlines. These style signals complement the vocabulary captured by TF-IDF. Verify the direction in the table above.

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(14,7))
for a, c in zip(ax.ravel(), ["exclam_count","caps_ratio","title_caps_ratio","quote_count","avg_word_len","question_count"]):
    sns.boxplot(data=style, x="label", y=c, ax=a, showfliers=False); a.set_title(c)
plt.tight_layout(); plt.savefig("../images/style_features.png", dpi=120); plt.show()

# 6. Train / Test Split & TF-IDF

In [ ]:
X_text = data["content"]; X_style = style.drop(columns="label").reset_index(drop=True); y = data["y"].reset_index(drop=True)
X_text = X_text.reset_index(drop=True)

idx_train, idx_test = train_test_split(np.arange(len(y)), test_size=0.2, stratify=y, random_state=RANDOM_STATE)
Xt_train, Xt_test = X_text.iloc[idx_train], X_text.iloc[idx_test]
y_train, y_test = y.iloc[idx_train], y.iloc[idx_test]

# Fit vectorizer on TRAIN ONLY (avoids leakage)
tfidf = TfidfVectorizer(stop_words="english", max_features=20000, ngram_range=(1,2), min_df=2 if len(y) > 1000 else 1, sublinear_tf=True)
Xtr = tfidf.fit_transform(Xt_train); Xte = tfidf.transform(Xt_test)
print("TF-IDF matrix:", Xtr.shape, "| train/test:", len(y_train), "/", len(y_test))

# Scaled style features (needed for distance/linear models if used alongside TF-IDF)
scaler = StandardScaler().fit(X_style.iloc[idx_train])
print("Class balance (train):", y_train.mean().round(3))

# 7. Model Development

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, C=1.0, random_state=RANDOM_STATE),
    "Decision Tree":       DecisionTreeClassifier(max_depth=20, random_state=RANDOM_STATE),
    "Random Forest":       RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE),
    "AdaBoost":            AdaBoostClassifier(n_estimators=200, random_state=RANDOM_STATE),
    "KNN":                 KNeighborsClassifier(n_neighbors=5, metric="cosine", algorithm="brute", n_jobs=-1),
}
def evaluate(model, Xa, ya, Xb, yb):
    p_a, p_b = model.predict(Xa), model.predict(Xb)
    return dict(train_acc=accuracy_score(ya, p_a), test_acc=accuracy_score(yb, p_b),
                precision=precision_score(yb, p_b), recall=recall_score(yb, p_b), f1=f1_score(yb, p_b))

results, fitted = {}, {}
for name, m in models.items():
    m.fit(Xtr, y_train); fitted[name] = m
    results[name] = evaluate(m, Xtr, y_train, Xte, y_test)
    if hasattr(m, "predict_proba"): results[name]["roc_auc"] = roc_auc_score(y_test, m.predict_proba(Xte)[:,1])
res = pd.DataFrame(results).T
res["gap(train-test acc)"] = res["train_acc"] - res["test_acc"]
res.round(4).sort_values("f1", ascending=False)

| Model | Strengths | Weaknesses |
|---|---|---|
| Logistic Regression | Fast, strong on sparse TF-IDF, coefficients are interpretable | Linear boundary only |
| Decision Tree | Easy to explain | Overfits; unstable on high-dim text |
| Random Forest | Robust, handles non-linearity | Slow, large, less interpretable on text |
| AdaBoost | Focuses on hard examples | Sensitive to noise, slower |
| KNN | No training, simple | Slow at inference, weak in high dimensions |

*(Update with your observed results.)*

# 8. Model Evaluation

In [ ]:
fig, ax = plt.subplots(1, 5, figsize=(22,4))
for a, (name, m) in zip(ax, fitted.items()):
    ConfusionMatrixDisplay(confusion_matrix(y_test, m.predict(Xte)), display_labels=["Real","Fake"]).plot(ax=a, colorbar=False, cmap="Blues")
    a.set_title(name)
plt.tight_layout(); plt.savefig("../images/confusion_matrices.png", dpi=120); plt.show()

**Which metric matters?** Treating *Fake* as the positive class:
- **Recall** = share of fake articles caught. A false negative lets misinformation through (reputational / societal harm).
- **Precision** = share of flagged articles that are truly fake. A false positive wrongly blocks or delays a legitimate article (editorial cost, trust with authors).
- **F1** balances both; accuracy alone can hide imbalance. For a *pre-publication flagging* tool with human review, prioritise **high recall with acceptable precision**.

In [ ]:
# Generalisation check: 5-fold CV on training set for the strongest candidates
cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
best_names = res.sort_values("f1", ascending=False).index[:3]
cv_rows = {}
for n in best_names:
    sc = cross_val_score(models[n].__class__(**models[n].get_params()), Xtr, y_train, cv=cv, scoring="f1", n_jobs=1)
    cv_rows[n] = dict(cv_f1_mean=sc.mean(), cv_f1_std=sc.std(), test_f1=res.loc[n, "f1"])
pd.DataFrame(cv_rows).T.round(4)

In [ ]:
# Optional: light hyper-parameter tuning of Logistic Regression
from sklearn.model_selection import GridSearchCV
gs = GridSearchCV(LogisticRegression(max_iter=1000, random_state=RANDOM_STATE), {"C": [0.1, 1, 10, 30]}, cv=3, scoring="f1")
gs.fit(Xtr, y_train); print("Best params:", gs.best_params_, "| CV F1:", round(gs.best_score_, 4))

## 8.1 Model selection
Select the final model by comparing **test F1/recall, the train–test gap (over/under-fitting), CV stability, inference speed and interpretability** — not accuracy alone. Write your choice and reasoning in the report. Logistic Regression on TF-IDF is a common winner here because it is accurate, fast and explainable; confirm with your own numbers.

In [ ]:
FINAL_NAME = "Logistic Regression"          # <- change if another model wins in your results
final = gs.best_estimator_ if FINAL_NAME == "Logistic Regression" else fitted[FINAL_NAME]
final.fit(Xtr, y_train)
pred = final.predict(Xte)
print(classification_report(y_test, pred, target_names=["Real","Fake"], digits=4))

# 9. Robustness: how much of the score is leakage?
We train the same classifier (a) on text **with** the agency tag left in and (b) on the cleaned text, to measure how much the shortcut inflates performance.

In [ ]:
def quick_lr(texts):
    xa, xb, ya, yb = train_test_split(texts, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
    v = TfidfVectorizer(stop_words="english", max_features=20000, sublinear_tf=True)
    m = LogisticRegression(max_iter=1000).fit(v.fit_transform(xa), ya)
    p = m.predict(v.transform(xb)); return f1_score(yb, p), recall_score(yb, p)

raw_variant = (data["title"] + " " + data["text"]).map(lambda t: clean_text(t, strip_agency=False)).reset_index(drop=True)
f1_raw, r_raw = quick_lr(raw_variant); f1_clean, r_clean = quick_lr(X_text)
pd.DataFrame({"with agency tag": [f1_raw, r_raw], "agency tag removed": [f1_clean, r_clean]}, index=["F1", "Recall"]).round(4)

# 10. Probability Scores & Threshold Recommendation

**Question:** *If a news platform deployed this model to flag articles before publishing, what confidence threshold would you recommend, and why?*

In [ ]:
proba = final.predict_proba(Xte)[:, 1]            # P(article is Fake)
prec, rec, thr = precision_recall_curve(y_test, proba)
plt.figure(figsize=(7,4.5))
plt.plot(thr, prec[:-1], label="Precision"); plt.plot(thr, rec[:-1], label="Recall")
plt.xlabel("Threshold on P(fake)"); plt.legend(); plt.title("Precision / Recall vs threshold"); plt.tight_layout()
plt.savefig("../images/threshold_curve.png", dpi=120); plt.show()

rows = []
for t in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95]:
    p = (proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, p).ravel()
    rows.append(dict(threshold=t, precision=precision_score(y_test, p, zero_division=0), recall=recall_score(y_test, p),
                     false_positives=fp, false_negatives=fn, flagged_pct=p.mean()))
thr_table = pd.DataFrame(rows).round(4); thr_table

**Recommended policy (three-band, human-in-the-loop):**
- **P(fake) ≥ ~0.80** → hold for editorial review (high precision band).
- **~0.40 – 0.80** → soft flag / second-opinion queue.
- **< ~0.40** → publish normally.

Reasoning: a missed fake is costly, so the lower band keeps recall high, while the upper band keeps false alarms low enough for editors to trust the tool. **Re-tune these cut-offs using the table above on your data** and the platform's review capacity; never auto-block on model score alone.

# 11. Interpretability — what drives the predictions?

In [ ]:
coef = final.coef_[0]; vocab = np.array(tfidf.get_feature_names_out())
top_fake = pd.DataFrame({"term": vocab[np.argsort(coef)[-20:][::-1]], "weight": np.sort(coef)[-20:][::-1]})
top_real = pd.DataFrame({"term": vocab[np.argsort(coef)[:20]], "weight": np.sort(coef)[:20]})
fig, ax = plt.subplots(1, 2, figsize=(13,6))
sns.barplot(data=top_fake, y="term", x="weight", ax=ax[0], color="indianred"); ax[0].set_title("Terms pushing toward FAKE")
sns.barplot(data=top_real, y="term", x="weight", ax=ax[1], color="steelblue"); ax[1].set_title("Terms pushing toward REAL")
plt.tight_layout(); plt.savefig("../images/top_coefficients.png", dpi=120); plt.show()

# 12. Save Artifacts

In [ ]:
joblib.dump({"model": final, "tfidf": tfidf}, "../models/fake_news_lr_tfidf.joblib")
res.round(4).to_csv("../reports/model_comparison.csv")
thr_table.to_csv("../reports/threshold_analysis.csv", index=False)

scored = pd.DataFrame({"content_preview": Xt_test.str[:120].values, "true_label": np.where(y_test==1, "Fake", "Real"),
                       "fake_probability": proba.round(4), "predicted": np.where(pred==1, "Fake", "Real")})
scored.to_csv("../reports/test_predictions_with_probability.csv", index=False)
scored.head()

In [ ]:
def predict_article(title, text, threshold=0.5):
    c = clean_text(title + " " + text)
    p = final.predict_proba(tfidf.transform([c]))[0, 1]
    return {"fake_probability": round(float(p), 4), "flag": "Fake" if p >= threshold else "Real"}

predict_article("Senate approves budget after lengthy debate", "Lawmakers said on Tuesday the bill would fund infrastructure projects over five years, according to a statement.")

# 13. Business Insights & Recommendations
*Complete with your findings from the full dataset.*

**Linguistic patterns of fake news** — see Section 11 coefficients and Section 3.3 word frequencies.
**Subject categories** — see Section 3.2; note whether subject overlap is too low to draw topic-level conclusions.
**Writing style** — see Section 5 table (caps, punctuation, length, quotes).
**Limitations of a text-only detector** — dataset is mostly US politics 2016–2017 (topic/time drift); publisher artefacts can leak the label; satire/opinion pieces are hard; adversaries can rewrite style; no fact-checking against reality.
**Signals to add** — source/domain credibility, author history, publication date & timing, social-sharing behaviour, claim verification against trusted knowledge bases, image/metadata checks.

**Recommendations**
1. Use the model as a *triage* tool with human review, not an auto-publisher/blocker.
2. Adopt the three-band threshold policy and revisit quarterly.
3. Retrain on fresh, multi-publisher data to counter drift and leakage.
4. Add source-credibility and metadata features before production.
5. Monitor false-positive complaints and missed fakes as live KPIs.